In [2]:
! pip install spacy rdflib pyshacl requests

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\sayil\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.6/12.8 MB 5.1 MB/s eta 0:00:03
     -------- ------------------------------- 2.6/12.8 MB 5.3 MB/s eta 0:00:02
     ----------- ---------------------------- 3.7/12.8 MB 5.2 MB/s eta 0:00:02
     -------------- ------------------------- 4.7/12.8 MB 5.2 MB/s eta 0:00:02
     ------------------ --------------------- 6.0/12.8 MB 5.2 MB/s eta 0:00:02
     ---------------------- ----------------- 7.3/12.8 MB 5.3 MB/s eta 0:00:02
     -------------------------- ------------- 8.4/12.8 MB 5.3 MB/s eta 0:00:01
     ---------------------------- ----------- 9.2/12.8 MB 5.2 MB/s eta 0:00:01
     -------------------------------- ------- 10.5/12.8 MB 5.2 MB/s eta 0:00:01
     ------------------------------------ --- 11.5/12.8 MB 5.2 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 5.2 M


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import requests, pathlib
HEADERS = {"User-Agent": "KRR-coursework/1.0 (your-university-email)"}
def fetch(title):
    r = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "prop": "extracts", "explaintext": 1,
        "redirects": 1, "titles": title, "format": "json"},
        headers=HEADERS, timeout=30)
    page = next(iter(r.json()["query"]["pages"].values()))
    return page["extract"]

pathlib.Path("data/raw").mkdir(parents=True, exist_ok=True)

titles = [
    "Arthur Compton",
    "Eugene Wigner",
    "John Bardeen",
    "Hans Bethe",
    "Albert A. Michelson",
]

for title in titles:
    text = fetch(title)
    filename = title.replace(" ", "_") + ".txt"
    pathlib.Path("data/raw", filename).write_text(text, encoding="utf-8")

In [5]:
import csv
import re
from pathlib import Path
import spacy

nlp = spacy.load("en_core_web_sm")

INPUT_DIR = Path("data/raw")
OUTPUT_FILE = Path("out/triples.csv")

PREDICATES = {
    "bornIn", "educatedAt", "receivedAward", "worksAt", "doctoralAdvisor"
}

ALIASES = {
    "Dr. Khokhar": "Rukhsana Khokhar",
    "Khokhar": "Rukhsana Khokhar",
    "Albert A. Michelson": "Albert Abraham Michelson",
    "Abraham Michelson": "Albert Abraham Michelson",
    "Michelson": "Albert Abraham Michelson",
    "Arthur Compton": "Arthur Holly Compton",
    "Compton": "Arthur Holly Compton",
    "Eugene Wigner": "Eugene Paul Wigner",
    "Paul Wigner": "Eugene Paul Wigner",
    "Wigner": "Eugene Paul Wigner",
    "Hans Bethe": "Hans Albrecht Eduard Bethe",
    "Hans Albrecht": "Hans Albrecht Eduard Bethe",
    "Eduard Bethe": "Hans Albrecht Eduard Bethe",
    "Bethe": "Hans Albrecht Eduard Bethe",
    "Bardeen": "John Bardeen",
    "John Robert Schrieffer": "John Bardeen",
}

AWARD_WORDS = {"prize", "award", "medal", "fellowship", "honor", "honour"}
DEGREE_WORDS = {"bachelor", "master", "ph.d", "phd", "doctorate", "degree"}

URL_RE = re.compile(r"\s*\(https?://[^()]*(?:\([^()]*\)[^()]*)*\)")
FOOTNOTE_RE = re.compile(r"\[\d+\]?")
YEAR_RE = re.compile(r"^\d{4}$")
LEADING_ARTICLE_RE = re.compile(r"^(?:the|a|an)\s+", re.IGNORECASE)
TRAILING_PUNCT_RE = re.compile(r"[\s:;,.\]]+\Z")
POSSESSIVE_RE = re.compile(r"['’]s$")

FEMALE_NAMES = {"Rukhsana", "she", "her", "hers"}
MALE_NAMES = {"Daniyal", "he", "him", "his"}

MERGE_COUNT = 0
LAST_PERSON = None
MALE_PERSON = None
FEMALE_PERSON = None


def reset_state():
    global MERGE_COUNT, LAST_PERSON, MALE_PERSON, FEMALE_PERSON
    MERGE_COUNT = 0
    LAST_PERSON = None
    MALE_PERSON = None
    FEMALE_PERSON = None


def read_text_safely(path: Path) -> str:
    raw = path.read_bytes()
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        text = raw.decode("latin-1")
    if any(ch in text for ch in "\u00c3\u00c2\u00d4"):
        try:
            text = text.encode("latin-1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    text = text.lstrip("\ufeff")
    text = URL_RE.sub("", text)
    return text


def clean(text: str) -> str:
    if not text:
        return text
    text = URL_RE.sub("", text)
    text = FOOTNOTE_RE.sub("", text)
    return text.strip()


def normalize(name: str) -> str:
    global MERGE_COUNT
    if not name:
        return name
    name = clean(name)
    for alias, original in ALIASES.items():
        if name.casefold() == alias.casefold():
            if name != original:
                MERGE_COUNT += 1
            return original
    return name


def strip_obj(text: str) -> str:
    text = TRAILING_PUNCT_RE.sub("", text or "")
    text = LEADING_ARTICLE_RE.sub("", text)
    return text.strip()


def is_negated(token) -> bool:
    for t in [token] + list(token.children):
        if t.dep_ == "neg" or t.lower_ in {"never", "not", "n't", "no"}:
            return True
    return False


def full_np(token) -> str:
    for ent in token.doc.ents:
        if ent.start <= token.i < ent.end:
            return clean(ent.text)
    for chunk in token.doc.noun_chunks:
        if chunk.start <= token.i < chunk.end:
            return clean(chunk.text)
    return clean(token.text)


def place_np(token) -> str:
    for ent in token.doc.ents:
        if ent.start <= token.i < ent.end and ent.label_ in {"GPE", "LOC", "FAC"}:
            return clean(ent.text)
    text = full_np(token)
    text = re.sub(r"\s+(?:in|at|on)\s+\d{4}.*$", "", text)
    return text


def is_award(text: str) -> bool:
    t = text.casefold()
    return any(w in t for w in AWARD_WORDS) and not any(w in t for w in DEGREE_WORDS)


def is_degree(text: str) -> bool:
    t = text.casefold()
    return any(w in t for w in DEGREE_WORDS)


def update_state_with_entities(doc):
    global LAST_PERSON, MALE_PERSON, FEMALE_PERSON
    for ent in doc.ents:
        if ent.label_ == "PERSON" and not is_award(ent.text):
            name = normalize(ent.text)
            LAST_PERSON = name
            if any(n in name for n in FEMALE_NAMES):
                FEMALE_PERSON = name
            elif any(n in name for n in MALE_NAMES):
                MALE_PERSON = name
            else:
                if "Rukhsana" in name:
                    FEMALE_PERSON = name
                else:
                    MALE_PERSON = name


def extract_doctoral_advisor(doc):
    triples = []
    for token in doc:
        if token.lemma_ == "be":
            nsubj = [w for w in token.children if w.dep_ == "nsubj"]
            attr = [w for w in token.children if w.dep_ in ("attr", "acomp")]
            if nsubj and attr:
                subj_np = nsubj[0]
                if "advisor" in subj_np.text.lower() or "adviser" in subj_np.text.lower():
                    person = None
                    for child in subj_np.children:
                        if child.dep_ == "poss":
                            person = full_np(child)
                    if person:
                        person = POSSESSIVE_RE.sub("", person)
                        advisor = full_np(attr[0])
                        if advisor:
                            triples.append((
                                normalize(person),
                                "doctoralAdvisor",
                                normalize(advisor),
                            ))
    return triples


def extract_sentence(doc):
    triples = []

    prev_male = MALE_PERSON
    prev_female = FEMALE_PERSON
    prev_last = LAST_PERSON

    subject = None

    for token in doc:
        if token.dep_ in {"nsubj", "nsubjpass"}:
            if token.lower_ in {"he", "his", "him"}:
                subject = prev_male or prev_last
                break
            if token.lower_ in {"she", "her", "hers"}:
                subject = prev_female or prev_last
                break

    if not subject:
        for token in doc:
            if token.dep_ in {"nsubj", "nsubjpass"} and token.ent_type_ == "PERSON":
                subject = full_np(token)
                break

    if not subject:
        subject = prev_last

    if not subject:
        update_state_with_entities(doc)
        return triples

    subject = POSSESSIVE_RE.sub("", subject)
    subject = normalize(subject)

    triples.extend(extract_doctoral_advisor(doc))

    for token in doc:
        if is_negated(token):
            continue
        lemma = token.lemma_.lower()

        if lemma == "bear":
            for child in token.children:
                if child.dep_ in {"prep", "agent"} and child.lower_ in {"in", "at"}:
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            place = TRAILING_PUNCT_RE.sub("", place_np(obj))
                            if place and not YEAR_RE.match(place):
                                triples.append((subject, "bornIn", normalize(place)))

        if lemma in {"study", "graduate", "attend"}:
            for child in token.children:
                if child.dep_ == "prep" and child.lower_ in {"at", "in", "from"}:
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            o = strip_obj(full_np(obj))
                            if o:
                                triples.append((subject, "educatedAt", normalize(o)))

        if lemma == "earn" and any(
            child.dep_ in {"dobj", "obj"} and is_degree(full_np(child))
            for child in token.children
        ):
            for child in token.children:
                if child.dep_ == "prep" and child.lower_ == "from":
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            o = strip_obj(full_np(obj))
                            if o:
                                triples.append((subject, "educatedAt", normalize(o)))

        if lemma in {"receive", "win", "earn", "award"}:
            for child in token.children:
                if child.dep_ in {"dobj", "obj", "attr", "nsubjpass"}:
                    o = strip_obj(full_np(child))
                    if is_award(o):
                        triples.append((subject, "receivedAward", normalize(o)))

        if lemma in {"join", "work", "employ"}:
            for child in token.children:
                if child.dep_ == "prep" and child.lower_ in {"at", "for"}:
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            o = strip_obj(full_np(obj))
                            if o and not YEAR_RE.match(o):
                                triples.append((subject, "worksAt", normalize(o)))
                if lemma == "join" and child.dep_ in {"dobj", "obj"}:
                    o = strip_obj(full_np(child))
                    if o and not YEAR_RE.match(o):
                        triples.append((subject, "worksAt", normalize(o)))
                if child.dep_ == "prep" and child.lower_ == "as":
                    for role in child.children:
                        for sub in role.children:
                            if sub.dep_ == "prep" and sub.lower_ == "at":
                                for obj in sub.children:
                                    if obj.dep_ == "pobj":
                                        o = strip_obj(full_np(obj))
                                        if o:
                                            triples.append((subject, "worksAt", normalize(o)))

        if lemma in {"professor", "researcher", "lecturer"}:
            for child in token.children:
                if child.dep_ == "prep" and child.lower_ in {"at", "in"}:
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            o = strip_obj(full_np(obj))
                            if o:
                                triples.append((subject, "worksAt", normalize(o)))

    grouped = {}
    for s, p, o in triples:
        grouped.setdefault((s, p), []).append(o)

    result = []
    for (s, p), objs in grouped.items():
        objs = sorted(set(objs), key=len, reverse=True)
        kept = []
        for o in objs:
            if not any(o.casefold() in k.casefold() for k in kept):
                kept.append(o)
        for o in kept:
            result.append((s, p, o))

    update_state_with_entities(doc)
    return result


def main():
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    all_rows = []

    for file_path in sorted(INPUT_DIR.glob("*.txt")):
        reset_state()
        text = read_text_safely(file_path)
        doc = nlp(text)

        for sent in doc.sents:
            triples = extract_sentence(sent.as_doc())
            for subject, predicate, obj in triples:
                if predicate in PREDICATES and subject and obj:
                    all_rows.append({
                        "subject": subject,
                        "predicate": predicate,
                        "object": obj,
                        "source_doc": file_path.name,
                        "sentence": clean(sent.text),
                    })

    columns = ["subject", "predicate", "object", "source_doc", "sentence"]
    with OUTPUT_FILE.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(all_rows)

    print("Extraction completed!")
    print("Total triples:", len(all_rows))
    print("Alias merges reported:", MERGE_COUNT)
    print("Output:", OUTPUT_FILE)


if __name__ == "__main__":
    main()

Extraction completed!
Total triples: 93
Alias merges reported: 3
Output: out\triples.csv


In [6]:
import csv, re
from pathlib import Path
from rdflib import Graph, Namespace, RDF, URIRef

EX = Namespace("http://example.org/")
CSV_IN  = Path("out/triples.csv")
GRAPH_OUT = Path("out/kg.ttl")

SUBJECT_PREDICATES = {"receivedAward", "worksAt", "bornIn", "educatedAt", "doctoralAdvisor"}

KNOWN_PERSONS = {
    "Arthur Holly Compton", "Eugene Paul Wigner", "John Bardeen",
    "Hans Albrecht Eduard Bethe", "Albert Abraham Michelson",
    "John Robert Schrieffer", "Brian Josephson", "Walter Houser Brattain",
    "Rukhsana Khokhar", "Daniyal Siddiqui",
}

NO_TYPE_OBJECTS = {"Nobel Prize in Physics twice"}

RANGE_CLASS = {
    "receivedAward":   EX.Award,
    "worksAt":         EX.Organization,
    "educatedAt":      EX.Organization,
    "bornIn":          EX.Place,
    "doctoralAdvisor": EX.Person,
}

def iri(text: str) -> URIRef:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return URIRef(EX[slug])

g = Graph()
g.bind("ex", EX)

with CSV_IN.open(encoding="utf-8") as f:
    for row in csv.DictReader(f):
        s = iri(row["subject"])
        o = iri(row["object"])
        p = EX[row["predicate"]]
        g.add((s, p, o))

        if row["predicate"] in SUBJECT_PREDICATES and row["subject"] in KNOWN_PERSONS:
            g.add((s, RDF.type, EX.Person))

        if row["object"] not in NO_TYPE_OBJECTS:
            g.add((o, RDF.type, RANGE_CLASS[row["predicate"]]))

GRAPH_OUT.parent.mkdir(parents=True, exist_ok=True)
GRAPH_OUT.write_text(g.serialize(format="turtle"), encoding="utf-8")
print(f"Wrote {GRAPH_OUT} with {len(g)} triples")

Wrote out\kg.ttl with 179 triples


In [7]:
from pathlib import Path

ONTOLOGY_TTL = """@prefix ex:   <http://example.org/> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex: a owl:Ontology ; rdfs:label "Physicists triples ontology" .

ex:Person       a owl:Class .
ex:Place        a owl:Class .
ex:Organization a owl:Class .
ex:Award        a owl:Class .

ex:Person owl:disjointWith ex:Place , ex:Organization , ex:Award .
ex:Place  owl:disjointWith ex:Organization , ex:Award .
ex:Organization owl:disjointWith ex:Award .

ex:receivedAward a owl:ObjectProperty ;
    rdfs:domain ex:Person ; rdfs:range ex:Award .

ex:worksAt a owl:ObjectProperty ;
    rdfs:domain ex:Person ; rdfs:range ex:Organization .

ex:bornIn a owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain ex:Person ; rdfs:range ex:Place .

ex:educatedAt a owl:ObjectProperty ;
    rdfs:domain ex:Person ; rdfs:range ex:Organization .

ex:doctoralAdvisor a owl:ObjectProperty ;
    rdfs:domain ex:Person ; rdfs:range ex:Person .
"""

SHAPES_TTL = """@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

ex:PersonShape a sh:NodeShape ;
    sh:targetClass ex:Person ;
    sh:property ex:BornInShape ;
    sh:property ex:WorksAtShape ;
    sh:property ex:ReceivedAwardShape ;
    sh:property ex:EducatedAtShape ;
    sh:property ex:DoctoralAdvisorShape .

ex:BornInShape a sh:PropertyShape ;
    sh:path ex:bornIn ;
    sh:maxCount 1 ;
    sh:class ex:Place ;
    sh:message "A person has at most one birthplace; it must be a Place." .

ex:WorksAtShape a sh:PropertyShape ;
    sh:path ex:worksAt ;
    sh:minCount 1 ;
    sh:class ex:Organization ;
    sh:message "A person must have at least one worksAt value, and it must be an Organization." .

ex:ReceivedAwardShape a sh:PropertyShape ;
    sh:path ex:receivedAward ;
    sh:class ex:Award ;
    sh:message "receivedAward values must be typed as ex:Award." .

ex:EducatedAtShape a sh:PropertyShape ;
    sh:path ex:educatedAt ;
    sh:class ex:Organization ;
    sh:message "educatedAt values must be Organizations." .

ex:DoctoralAdvisorShape a sh:PropertyShape ;
    sh:path ex:doctoralAdvisor ;
    sh:class ex:Person ;
    sh:message "doctoralAdvisor values must be Persons." .
"""

Path("ontology.ttl").write_text(ONTOLOGY_TTL, encoding="utf-8")
Path("shapes.ttl").write_text(SHAPES_TTL, encoding="utf-8")
print("Wrote ontology.ttl and shapes.ttl")

Wrote ontology.ttl and shapes.ttl


In [8]:
from pathlib import Path
from rdflib import Graph
from pyshacl import validate

data   = Graph().parse("out/kg.ttl",   format="turtle")
shapes = Graph().parse("shapes.ttl",   format="turtle")
onto   = Graph().parse("ontology.ttl", format="turtle")

conforms, _, report = validate(
    data_graph=data, shacl_graph=shapes, ont_graph=onto, inference="rdfs",
)

Path("shacl_report.txt").write_text(report, encoding="utf-8")
print("Conforms:", conforms)
print(report)

Conforms: False
Validation Report
Conforms: False
Results (14):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: ex:WorksAtShape
	Focus Node: ex:Abraham_Kuhn
	Result Path: ex:worksAt
	Message: A person must have at least one worksAt value, and it must be an Organization.
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: ex:WorksAtShape
	Focus Node: ex:Anderson_Professor
	Result Path: ex:worksAt
	Message: A person must have at least one worksAt value, and it must be an Organization.
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: ex:WorksAtShape
	Focus Node: ex:Bar_Mitzvah
	Result Path: ex:worksAt
	Message: A person must have at least one worksAt value, and it must be an Organization.
Constra

In [7]:
import pandas as pd, json

def pick(path, positions, label):
    df = pd.read_csv(path)
    df = df.rename(columns={"person": "subject"}) \
           .sort_values(["subject", "predicate", "object"]) \
           .reset_index(drop=True)
    n = len(df)
    print(f"\n=== {label}  (list length = {n}) ===")
    picked = []
    for p in positions:
        idx = (p - 1) % n
        row = df.iloc[idx].to_dict()
        row["_position"] = p
        row["_wrapped"]  = p > n
        picked.append(row)
        print(f"\n-- {label} #{p}  (row {idx+1}/{n}{' — WRAPPED' if p>n else ''}) --")
        for k, v in row.items():
            if not k.startswith("_"):
                print(f"  {k}: {v}")
    return picked

fp = pick("eval/false_positives.csv", [1, 8, 11, 17], "FP")
fn = pick("eval/false_negatives.csv", [2, 3, 8, 15], "FN")

json.dump({"fp": fp, "fn": fn}, open("eval/_chosen_items.json","w"),
          indent=2, default=str)
print("\n-> eval/_chosen_items.json")


=== FP  (list length = 16) ===

-- FP #1  (row 1/16) --
  subject: Albert A. Michelson
  predicate: bornIn
  object: Strelno
  source_doc: Albert_A._Michelson.txt
  sentence: == Biography ==


=== Early life ===
Albert Abraham Michelson was born on December 19, 1852, in Strelno (now Strzelno, Poland), Prussia, into a Polish-Jewish family.

-- FP #8  (row 8/16) --
  subject: Arthur Compton
  predicate: worksAt
  object: Oxford University
  source_doc: Arthur_Compton.txt
  sentence: In 1934, he returned to England as Eastman visiting professor at Oxford University.

-- FP #11  (row 11/16) --
  subject: Eugene Wigner
  predicate: educatedAt
  object: 1920
  source_doc: Eugene_Wigner.txt
  sentence: After graduating from the secondary school in 1920, Wigner enrolled at the Budapest University of Technical Sciences, known as the Műegyetem.

-- FP #17  (row 1/16 — WRAPPED) --
  subject: Albert A. Michelson
  predicate: bornIn
  object: Strelno
  source_doc: Albert_A._Michelson.txt
  sentenc

In [10]:
import requests, time, re
from pathlib import Path
import pandas as pd

HEADERS   = {"User-Agent": "KRR-coursework/1.0 (your-university-email)"}
WD_API    = "https://www.wikidata.org/w/api.php"
WD_ENTITY = "https://www.wikidata.org/wiki/Special:EntityData/{}.json"

PROPS = {
    "bornIn":          "P19",
    "educatedAt":      "P69",
    "receivedAward":   "P166",
    "worksAt":         "P108",
    "doctoralAdvisor": "P184",
}

PEOPLE = ["Arthur Compton", "Eugene Wigner", "John Bardeen",
          "Hans Bethe", "Albert A. Michelson"]

SUBJECT_MAP = {
    "Arthur Holly Compton":       "Arthur Compton",
    "Eugene Paul Wigner":         "Eugene Wigner",
    "John Bardeen":               "John Bardeen",
    "Hans Albrecht Eduard Bethe": "Hans Bethe",
    "Albert Abraham Michelson":   "Albert A. Michelson",
}

Path("eval").mkdir(exist_ok=True)

def search_qid(name):
    r = requests.get(WD_API, params={
        "action": "wbsearchentities", "search": name,
        "language": "en", "format": "json", "limit": 1,
    }, headers=HEADERS, timeout=30)
    hits = r.json().get("search", [])
    return hits[0]["id"] if hits else None

_label_cache = {}
def qid_label(qid):
    if qid in _label_cache:
        return _label_cache[qid]
    r = requests.get(WD_ENTITY.format(qid), headers=HEADERS, timeout=30)
    e = r.json()["entities"][qid]
    lbl = e["labels"].get("en", {}).get("value", qid)
    _label_cache[qid] = lbl
    return lbl

def claims(entity, prop):
    out = []
    for c in entity.get("claims", {}).get(prop, []):
        snak = c.get("mainsnak", {})
        if snak.get("snaktype") != "value":
            continue
        dv = snak.get("datavalue", {})
        if dv.get("type") == "wikibase-entityid":
            out.append(qid_label(dv["value"]["id"]))
        elif dv.get("type") == "string":
            out.append(dv["value"])
    return out

wd_rows = []
for name in PEOPLE:
    qid = search_qid(name)
    if not qid:
        print(f"[skip] no Wikidata QID for {name}"); continue
    ent = requests.get(WD_ENTITY.format(qid), headers=HEADERS, timeout=30)\
                 .json()["entities"][qid]
    for pred, prop in PROPS.items():
        for v in claims(ent, prop):
            wd_rows.append({"person": name, "qid": qid,
                            "predicate": pred, "object": v})
    time.sleep(0.5)

wd_df = pd.DataFrame(wd_rows)
wd_df.to_csv("eval/wikidata_all.csv", index=False)
print(f"Wikidata: {len(wd_df)} rows -> eval/wikidata_all.csv")

for name in PEOPLE:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    wd_df[wd_df["person"] == name].to_csv(f"eval/wikidata_{slug}.csv", index=False)

tri = pd.read_csv("out/triples.csv")
tri["canon_person"] = tri["subject"].map(SUBJECT_MAP)
tri = tri.dropna(subset=["canon_person"])

def norm(s):
    return re.sub(r"[^a-z0-9 ]+", "", str(s).casefold()).strip()

def same(a, b):
    a, b = norm(a), norm(b)
    if not a or not b:
        return False
    if a == b or a in b or b in a:
        return True
    ta, tb = set(a.split()), set(b.split())
    if not ta or not tb:
        return False
    return len(ta & tb) / max(len(ta), len(tb)) >= 0.6

metrics, fps, fns = [], [], []

for person in PEOPLE:
    ours = tri[tri["canon_person"] == person]
    wd   = wd_df[wd_df["person"] == person]

    for pred in PROPS:
        our_vals = sorted(set(ours[ours["predicate"] == pred]["object"].astype(str)))
        wd_vals  = sorted(set(wd[wd["predicate"] == pred]["object"].astype(str)))

        used_wd, tp_pairs = set(), []
        for ov in our_vals:
            for wv in wd_vals:
                if wv in used_wd:
                    continue
                if same(ov, wv):
                    used_wd.add(wv)
                    tp_pairs.append((ov, wv))
                    break

        matched_our = {p[0] for p in tp_pairs}
        tp = len(tp_pairs)
        fp = len(our_vals) - tp
        fn = len(wd_vals) - tp
        precision = tp / (tp + fp) if tp + fp else (0.0 if fn else 1.0)
        recall    = tp / (tp + fn) if tp + fn else (0.0 if fp else 1.0)

        metrics.append({
            "person": person, "predicate": pred,
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 3), "recall": round(recall, 3),
        })

        for ov in our_vals:
            if ov not in matched_our:
                row = ours[(ours["predicate"] == pred) & (ours["object"].astype(str) == ov)].iloc[0]
                fps.append({
                    "person": person, "predicate": pred, "object": ov,
                    "source_doc": row["source_doc"], "sentence": row["sentence"],
                })
        for wv in wd_vals:
            if wv not in used_wd:
                fns.append({"person": person, "predicate": pred, "object": wv})

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("eval/metrics.csv", index=False)

fp_df = pd.DataFrame(fps).sort_values(["person","predicate","object"]).reset_index(drop=True)
fn_df = pd.DataFrame(fns).sort_values(["person","predicate","object"]).reset_index(drop=True)
fp_df.to_csv("eval/false_positives.csv", index=False)
fn_df.to_csv("eval/false_negatives.csv", index=False)

tot_tp = metrics_df["tp"].sum(); tot_fp = metrics_df["fp"].sum(); tot_fn = metrics_df["fn"].sum()
print(f"\nMICRO  P = {tot_tp/(tot_tp+tot_fp):.3f}  "
      f"R = {tot_tp/(tot_tp+tot_fn):.3f}  "
      f"(TP={tot_tp}, FP={tot_fp}, FN={tot_fn})")
print(metrics_df.pivot_table(index="predicate",
                             values=["tp","fp","fn","precision","recall"],
                             aggfunc="sum"))
print(f"\nFP rows: {len(fp_df)}  FN rows: {len(fn_df)}")

Wikidata: 106 rows -> eval/wikidata_all.csv

MICRO  P = 0.407  R = 0.107  (TP=11, FP=16, FN=92)
                 fn  fp  precision  recall  tp
predicate                                     
bornIn            2   1      2.000   2.000   2
doctoralAdvisor   5   0      1.000   1.000   0
educatedAt       16   3      1.000   0.167   1
receivedAward    58   3      3.333   0.269   4
worksAt          11   9      1.700   1.500   4

FP rows: 16  FN rows: 92


In [9]:
import pandas as pd, json

def pick(path, positions, label):
    df = pd.read_csv(path)
    df = df.rename(columns={"person": "subject"}) \
           .sort_values(["subject", "predicate", "object"]) \
           .reset_index(drop=True)
    n = len(df)
    print(f"\n=== {label}  (list length = {n}) ===")
    picked = []
    for p in positions:
        idx = (p - 1) % n
        row = df.iloc[idx].to_dict()
        row["_position"] = p
        row["_wrapped"]  = p > n
        picked.append(row)
        print(f"\n-- {label} #{p}  (row {idx+1}/{n}{' — WRAPPED' if p>n else ''}) --")
        for k, v in row.items():
            if not k.startswith("_"):
                print(f"  {k}: {v}")
    return picked

fp = pick("eval/false_positives.csv", [1, 8, 11, 17], "FP")
fn = pick("eval/false_negatives.csv", [2, 3, 8, 15], "FN")

json.dump({"fp": fp, "fn": fn}, open("eval/_chosen_items.json","w"),
          indent=2, default=str)
print("\n-> eval/_chosen_items.json")


=== FP  (list length = 16) ===

-- FP #1  (row 1/16) --
  subject: Albert A. Michelson
  predicate: bornIn
  object: Strelno
  source_doc: Albert_A._Michelson.txt
  sentence: == Biography ==


=== Early life ===
Albert Abraham Michelson was born on December 19, 1852, in Strelno (now Strzelno, Poland), Prussia, into a Polish-Jewish family.

-- FP #8  (row 8/16) --
  subject: Arthur Compton
  predicate: worksAt
  object: Oxford University
  source_doc: Arthur_Compton.txt
  sentence: In 1934, he returned to England as Eastman visiting professor at Oxford University.

-- FP #11  (row 11/16) --
  subject: Eugene Wigner
  predicate: educatedAt
  object: 1920
  source_doc: Eugene_Wigner.txt
  sentence: After graduating from the secondary school in 1920, Wigner enrolled at the Budapest University of Technical Sciences, known as the Műegyetem.

-- FP #17  (row 1/16 — WRAPPED) --
  subject: Albert A. Michelson
  predicate: bornIn
  object: Strelno
  source_doc: Albert_A._Michelson.txt
  sentenc